# 🥉 Capa Bronce: Generación de Datos Sintéticos (AML)

Este notebook genera de manera distribuida transacciones y perfiles de clientes simulando comportamientos legítimos y de lavado de activos (AML).
Se utilizan UDFs de Pandas y PySpark para lograr una generación eficiente y productiva, guardando los resultados directamente en tablas Delta en el catálogo `workspace.aml_proyect`.

In [0]:
import uuid
import random
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, pandas_udf, explode
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

# Configuraciones Globales
UMBRAL_ALERTA = 10000.0
PROBABILIDAD_RUIDO = 0.05
MONEDA_ESTANDAR = 'USD'
TASAS_CAMBIO = {
    'USD': 1.00,
    'PEN': 3.35,
    'EUR': 0.86,
    'GBP': 0.74,
    'JPY': 159.35,
    'CNY': 6.72
}
TOTAL_REGISTROS_MIN = 2500000
TOTAL_REGISTROS_MAX = 3000000


In [0]:
class PerfilFinanciero:
    def __init__(self):
        self.pk_cliente = str(uuid.uuid4())
        self.edad = random.randint(18, 70)
        self.ocupacion_declarada = "Sin Declarar"
        self.ingreso_mensual_aprox = 0.0

    def obtener_dim_cliente(self) -> dict:
        return {
            "PK_Cliente": self.pk_cliente,
            "Edad": self.edad,
            "Ocupacion_Declarada": self.ocupacion_declarada,
            "Ingreso_Mensual_Aprox": float(round(self.ingreso_mensual_aprox, 2))
        }

    def _crear_transaccion(self, fk_origen: str, fk_destino: str, fecha: datetime, 
                           canal: str, geografia: str, monto: float) -> dict:
        moneda = random.choice(list(TASAS_CAMBIO.keys()))
        tasa = TASAS_CAMBIO[moneda]
        monto_original = monto * tasa
        return {
            "PK_Transaccion": str(uuid.uuid4()),
            "FK_Cliente_Origen": fk_origen,
            "FK_Cliente_Destino": fk_destino,
            "FK_Tiempo": fecha.strftime("%Y-%m-%d %H:%M:%S"),
            "FK_Canal": canal,
            "FK_Geografia": geografia,
            "Monto_Original": float(round(monto_original, 2)),
            "Moneda": moneda,
            "Monto_USD": float(round(monto, 2))
        }

    def _generar_fecha_legitima(self, fecha_base: datetime, es_salario: bool = False) -> Tuple[datetime, bool]:
        if es_salario:
            dias_quincena = [fecha_base + timedelta(days=i) for i in range(60) if (fecha_base + timedelta(days=i)).day in [15, 30, 31]]
            if not dias_quincena:
                dias_quincena = [fecha_base + timedelta(days=15)]
            fecha = random.choice(dias_quincena)
        else:
            fecha = fecha_base + timedelta(days=random.randint(0, 59))
        
        es_oficina = random.random() < 0.70
        hora = random.randint(9, 18) if es_oficina else random.choice(list(range(0, 9)) + list(range(19, 24)))
        fecha = fecha.replace(hour=hora, minute=random.randint(0, 59), second=random.randint(0, 59))
        return fecha, fecha.weekday() >= 5

    def _generar_fecha_sospechosa(self, fecha_base: datetime, rafaga_anterior: datetime = None) -> datetime:
        if rafaga_anterior:
            return rafaga_anterior + timedelta(seconds=random.randint(0, 59))
        fecha = fecha_base + timedelta(days=random.randint(0, 59))
        return fecha.replace(hour=random.randint(1, 5), minute=random.randint(0, 59), second=random.randint(0, 59))

class ClienteNormal(PerfilFinanciero):
    def __init__(self):
        super().__init__()
        self.ocupacion_declarada = random.choice(["Ingeniero", "Profesor", "Médico", "Comerciante", "Administrativo"])
        self.ingreso_mensual_aprox = random.uniform(1500, 5000)

    def generar_transacciones(self, fecha_inicio: datetime) -> List[dict]:
        transacciones = []
        fecha_salario, _ = self._generar_fecha_legitima(fecha_inicio, es_salario=True)
        transacciones.append(self._crear_transaccion(str(uuid.uuid4()), self.pk_cliente, fecha_salario, "Transferencia_Bancaria", "Local", self.ingreso_mensual_aprox))
        for _ in range(random.randint(10, 30)):
            fecha_compra, es_fin_semana = self._generar_fecha_legitima(fecha_inicio, es_salario=False)
            monto = random.uniform(5, 50) if es_fin_semana else random.uniform(10, 200)
            transacciones.append(self._crear_transaccion(self.pk_cliente, str(uuid.uuid4()), fecha_compra, "Tarjeta_Credito", "Local", monto))
        return transacciones

class PitufoBancario(PerfilFinanciero):
    def __init__(self):
        super().__init__()
        self.ocupacion_declarada = "Comerciante_Independiente"
        self.ingreso_mensual_aprox = random.uniform(1500, 3000)

    def generar_transacciones(self, fecha_inicio: datetime) -> List[dict]:
        transacciones = []
        num_transacciones = random.randint(3, 5)
        while True:
            montos = [random.uniform(2500.0, 9500.0) for _ in range(num_transacciones)]
            if sum(montos) > 10000.0: break
        dias_camuflaje = [fecha_inicio + timedelta(days=i) for i in range(60) if (fecha_inicio + timedelta(days=i)).day in [1, 15, 30, 31]]
        if not dias_camuflaje: dias_camuflaje = [fecha_inicio + timedelta(days=15)]
        fecha_base = random.choice(dias_camuflaje)
        fechas_tx = sorted([fecha_base.replace(hour=random.randint(12, 13) if random.random() < 0.5 else random.randint(17, 18), minute=random.randint(0, 59), second=random.randint(0, 59)) for _ in range(num_transacciones)])
        for f, m in zip(fechas_tx, montos):
            transacciones.append(self._crear_transaccion(self.pk_cliente, self.pk_cliente, f, "Deposito_Efectivo", "Cajero_Automatico", m))
        return transacciones

class LavadorPlataformas(PerfilFinanciero):
    def __init__(self):
        super().__init__()
        self.ocupacion_declarada = "Creador_de_Contenido"
        self.ingreso_mensual_aprox = random.uniform(500, 1000)

    def generar_transacciones(self, fecha_inicio: datetime) -> List[dict]:
        transacciones, monto_total = [], 0.0
        fecha_actual = None
        for _ in range(random.randint(100, 200)):
            fecha_actual = self._generar_fecha_sospechosa(fecha_inicio, fecha_actual)
            monto = random.uniform(50, 200)
            monto_total += monto
            transacciones.append(self._crear_transaccion(str(uuid.uuid4()), self.pk_cliente, fecha_actual, "Plataforma_Terceros", "Internacional", monto))
        transacciones.append(self._crear_transaccion(self.pk_cliente, str(uuid.uuid4()), self._generar_fecha_sospechosa(fecha_inicio, fecha_actual), "Transferencia_Bancaria", "Local", min(monto_total * random.uniform(0.90, 0.95), UMBRAL_ALERTA - 1.0)))
        return transacciones

class LavadorCrypto(PerfilFinanciero):
    def __init__(self):
        super().__init__()
        self.ocupacion_declarada = "Inversionista"
        self.ingreso_mensual_aprox = random.uniform(2000, 5000)

    def generar_transacciones(self, fecha_inicio: datetime) -> List[dict]:
        transacciones, monto_total = [], 0.0
        fecha_actual = None
        for _ in range(random.randint(5, 10)):
            fecha_actual = self._generar_fecha_sospechosa(fecha_inicio, fecha_actual)
            monto = random.uniform(500, 2000)
            monto_total += monto
            transacciones.append(self._crear_transaccion(str(uuid.uuid4()), self.pk_cliente, fecha_actual, "Crypto_Wallet", "Internacional", monto))
        exchange_uuid, monto_restante = str(uuid.uuid4()), monto_total
        while monto_restante > 0:
            fecha_actual = self._generar_fecha_sospechosa(fecha_inicio, fecha_actual)
            monto_salida = min(monto_restante, random.uniform(1000, 2500))
            monto_restante -= monto_salida
            transacciones.append(self._crear_transaccion(self.pk_cliente, exchange_uuid, fecha_actual, "Exchange_Crypto", "Internacional", monto_salida))
        return transacciones


In [0]:
def inyectar_ruido_pandas(df: pd.DataFrame) -> pd.DataFrame:
    df_ruido = df.copy()
    if 'Moneda' not in df_ruido.columns: df_ruido['Moneda'] = 'USD'
    mask = np.random.rand(len(df_ruido)) < PROBABILIDAD_RUIDO
    idx = df_ruido.index[mask].tolist()
    if not idx: return df_ruido
    np.random.shuffle(idx)
    splits = np.array_split(idx, 4)
    idx_f, idx_t, idx_n, idx_d = splits[0], splits[1], splits[2], splits[3]
    
    if len(idx_f) > 0:
        try:
            df_ruido.loc[idx_f, 'FK_Tiempo'] = pd.to_datetime(df_ruido.loc[idx_f, 'FK_Tiempo'], errors='coerce').dt.strftime('%d/%m/%Y %H:%M:%S')
        except: pass
    if len(idx_t) > 0:
        df_ruido.loc[idx_t, 'Moneda'] = np.random.choice(['usd', ' UDS ', 'US $', 'soles', ' pen ', 'eur ', ' GBP', 'yenes', 'cny', 'rmb '], size=len(idx_t))
    if len(idx_n) > 0:
        df_n = df_ruido.loc[idx_n].copy()
        df_n['PK_Transaccion'] = np.nan
        df_ruido = pd.concat([df_ruido, df_n], ignore_index=True)
    if len(idx_d) > 0:
        df_a = df_ruido.loc[idx_d].copy()
        try:
            df_a['FK_Tiempo'] = (pd.to_datetime(df_a['FK_Tiempo'], errors='coerce') + pd.Timedelta(seconds=1)).dt.strftime('%Y-%m-%d %H:%M:%S')
            df_ruido = pd.concat([df_ruido, df_a], ignore_index=True)
        except: pass
    return df_ruido


In [0]:
print("Iniciando generación de datos...")
# En lugar de generar millones en el nodo driver, usamos paralelismo de Spark
# Vamos a generar los perfiles de clientes en el driver (son pocos miles) y luego distribuimos la generación de transacciones
meta_registros = random.randint(TOTAL_REGISTROS_MIN, TOTAL_REGISTROS_MAX)
num_clientes_aprox = int(meta_registros / 22) # Aprox 22 tx promedio por cliente (mayormente ClienteNormal)

prob_sospechosos = random.uniform(0.002, 0.01)
prob_normal = 1.0 - prob_sospechosos
prob_por_lavador = prob_sospechosos / 3.0

clases_perfiles = [ClienteNormal, PitufoBancario, LavadorPlataformas, LavadorCrypto]
pesos = [prob_normal, prob_por_lavador, prob_por_lavador, prob_por_lavador]

clientes = []
transacciones_bruto = []
fecha_inicio_base = datetime.now() - timedelta(days=60)

# Generamos en lotes en el driver y paralelizamos
for _ in range(num_clientes_aprox):
    perfil_class = random.choices(clases_perfiles, weights=pesos, k=1)[0]
    perfil = perfil_class()
    clientes.append(perfil.obtener_dim_cliente())
    transacciones_bruto.extend(perfil.generar_transacciones(fecha_inicio_base))

df_clientes = spark.createDataFrame(clientes)
df_tx = spark.createDataFrame(transacciones_bruto)

# Inyectamos ruido distribuido usando applyInPandas / mapInPandas
def inyectar_ruido_iterator(iterator):
    for pdf in iterator:
        yield inyectar_ruido_pandas(pdf)

tx_schema = StructType([
    StructField("PK_Transaccion", StringType(), True),
    StructField("FK_Cliente_Origen", StringType(), True),
    StructField("FK_Cliente_Destino", StringType(), True),
    StructField("FK_Tiempo", StringType(), True),
    StructField("FK_Canal", StringType(), True),
    StructField("FK_Geografia", StringType(), True),
    StructField("Monto_Original", DoubleType(), True),
    StructField("Moneda", StringType(), True),
    StructField("Monto_USD", DoubleType(), True)
])

df_tx_ruido = df_tx.mapInPandas(inyectar_ruido_iterator, schema=tx_schema)


In [0]:
print("Escribiendo a Delta Lake...")

# Guardar Dimension Cliente
df_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.aml_proyect.clientes_bronce")

# Guardar Tabla de Hechos (Transacciones)
df_tx_ruido.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.aml_proyect.transacciones_bronce")

print("Datos Bronce generados y almacenados exitosamente.")
display(spark.sql("SELECT * FROM workspace.aml_proyect.transacciones_bronce LIMIT 5"))